# 03 — The YOLOv8 damage detector

Classification answers "**what** is in this image". Detection answers "**what AND where**". The output is a list of bounding boxes, each with a class label and a confidence score.

## Roadmap

1. What is a bounding box? (xyxy vs xywh, normalised vs pixel)
2. **IoU** (Intersection-over-Union) — with diagram + math
3. **NMS** (Non-Max Suppression) — how YOLO picks the best box
4. **Anchor-free prediction** — what YOLOv8 actually outputs at every grid cell
5. **mAP** — the evaluation metric, step by step
6. Runnable demo-scale training
7. Visualise predictions on a real image


In [ ]:
# === Colab bootstrap ===
# Safe to re-run. On a local clone with `pip install -e .` already done this
# is a no-op; on Colab it clones the repo + installs deps the first time.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/theDocWho/car-crash-fix-amount-predictor.git"
REPO_DIR = Path("car-crash-fix-amount-predictor")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

# Make `ccdp` importable whether or not the package was installed editable.
src_path = Path("src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("ccdp path:", src_path)
print("running in Colab:", IN_COLAB)


## 1. Bounding-box conventions

A box can be written as:

- **xyxy**: `(x1, y1, x2, y2)` — top-left and bottom-right corners.
- **xywh**: `(xc, yc, w, h)` — center + width + height.

Both can be in **absolute pixel** coordinates or **normalised** to `[0, 1]` relative to image size.

YOLO trains and predicts in **xywh-normalised**. Our `DetectedBox.xywh_norm` follows the same convention.


In [ ]:
def xywh_to_xyxy(xc, yc, w, h):
    return (xc - w/2, yc - h/2, xc + w/2, yc + h/2)

print(xywh_to_xyxy(0.5, 0.5, 0.4, 0.6))   # normalised


## 2. IoU — Intersection over Union

How "close" are two boxes? IoU is the standard answer:

$$
\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

- 1.0 means perfect overlap.
- 0.0 means disjoint.
- 0.5 is the typical threshold for "this prediction matches that ground-truth box".


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def iou(a, b):
    """IoU for boxes in xyxy form."""
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)

box_gt   = (1, 1, 5, 5)        # ground truth
box_pred = (3, 2, 7, 6)        # prediction
print(f"IoU = {iou(box_gt, box_pred):.3f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(0, 8); ax.set_ylim(0, 8); ax.set_aspect("equal")
ax.add_patch(patches.Rectangle((1,1), 4, 4, fill=False, edgecolor="green", lw=2, label="ground truth"))
ax.add_patch(patches.Rectangle((3,2), 4, 4, fill=False, edgecolor="red",   lw=2, label="prediction"))
ax.legend(); ax.grid(True)
plt.title(f"IoU = {iou(box_gt, box_pred):.3f}")
plt.show()


## 3. NMS — Non-Max Suppression

YOLO predicts *lots* of boxes per image — often several overlapping boxes around the same dent. NMS keeps the most confident one and suppresses the rest:

```
1. Sort all predictions by confidence (descending).
2. Take the top box; add it to the keep list.
3. Discard all remaining boxes that overlap it with IoU > threshold (e.g. 0.5).
4. Repeat until no boxes left.
```

This is **per-class** — a dent box doesn't suppress a scratch box even if they overlap.


In [ ]:
def nms(boxes, scores, iou_thresh=0.5):
    """Boxes: list of xyxy. Returns indices to keep."""
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) < iou_thresh]
    return keep

# 3 overlapping detections of the same dent at different confidences.
boxes = [(2,2,6,6), (2.5,2.5,6.5,6.5), (2.2,1.8,6.2,5.8)]
scores = [0.7, 0.9, 0.6]
print("kept indices:", nms(boxes, scores))
print("→ NMS keeps box index 1 (the most-confident one); discards 0 and 2.")


## 4. What does YOLOv8 actually predict?

YOLOv8 is **anchor-free**. The image is divided into a grid (e.g. 80×80 at one of three scales). At every grid cell, the model predicts:

- 4 numbers for the box offset (using a clever *distribution-focal-loss* encoding)
- 1 *objectness* score (is there anything here?)
- $C$ class probabilities (one per damage type)

So total output channels = `4 + 1 + C`. After NMS, the surviving cells become the final detections.


In [ ]:
# Use the project's own helper to peek at what the model outputs.
from ccdp.data.schema import DAMAGE_TYPES
C = len(DAMAGE_TYPES)
print(f"Per grid cell, YOLOv8 outputs {4 + 1 + C} channels:")
print(f"  4 (box) + 1 (objectness) + {C} (one per class)")


## 5. mAP — mean Average Precision

The detection metric. For one class:

1. Sort all predictions by confidence descending.
2. Walk down the list. Each prediction is a **True Positive** if its IoU with an unmatched ground-truth box ≥ 0.5, else a **False Positive**.
3. After each step, compute precision $= TP / (TP + FP)$ and recall $= TP / N_\text{gt}$.
4. Plot the precision-recall curve. The area under it is **AP** (Average Precision) for this class.
5. Average AP over all classes → **mAP**.

`mAP@0.5` means IoU threshold 0.5. `mAP@0.5:0.95` averages mAP across IoU thresholds 0.5, 0.55, …, 0.95 — much stricter.


In [ ]:
# Tiny worked example: one class, 5 predictions vs 3 ground-truth boxes.
import numpy as np

preds = [
    # (conf, is_TP)
    (0.95, True),
    (0.90, True),
    (0.85, False),
    (0.60, True),
    (0.30, False),
]
N_gt = 3
tp = fp = 0
prec_list, rec_list = [], []
for conf, is_tp in preds:
    if is_tp: tp += 1
    else:     fp += 1
    prec_list.append(tp / (tp + fp))
    rec_list.append(tp / N_gt)

print("Recall :", [round(r,2) for r in rec_list])
print("Precision:", [round(p,2) for p in prec_list])

import matplotlib.pyplot as plt
plt.figure(figsize=(5,4))
plt.plot(rec_list, prec_list, marker="o")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"AP (trapezoidal AUC) ≈ {np.trapz(prec_list, rec_list):.3f}")
plt.grid(True); plt.xlim(0,1); plt.ylim(0,1.05)
plt.show()


## 6. Demo-scale training

Ultralytics' YOLOv8 has its own trainer. Tiny epoch count below — replace `epochs=1, imgsz=320` with `epochs=50, imgsz=640` and a real dataset YAML for production. (The project's full-training entrypoint is `ccdp.train.detector`.)


In [ ]:
# Tiny "does it run?" smoke test — Ultralytics has a built-in `coco8` toy
# dataset that's perfect for verifying the training loop in seconds.
from ultralytics import YOLO

model = YOLO("yolov8n.yaml")        # nano, untrained
# Comment out next line if you don't want to download the toy dataset on Colab.
# results = model.train(data="coco8.yaml", epochs=1, imgsz=320, verbose=False)

print("ultralytics imported OK. Uncomment the train line to actually train (~30s).")


## 7. Visualise predictions on a real image

If the production detector is available locally (after `app.py` first boot or after running the production-weights download), we can run it end-to-end and draw the boxes.


In [ ]:
from pathlib import Path
from PIL import Image
from ccdp.viz import annotate_prediction

# This block only runs if production weights are present.
try:
    from ccdp.infer.variant_b import VariantBPipeline
    pipe = VariantBPipeline()
    sample = next(Path("data").rglob("*.jpg"), None) or next(Path("data").rglob("*.png"), None)
    if sample:
        img = Image.open(sample).convert("RGB")
        pred = pipe.predict(img)
        annotated = annotate_prediction(img, pred)
        annotated
    else:
        print("No sample image found under data/. Upload one to see real boxes.")
except Exception as e:
    print(f"Detector not available yet: {e}")


**Next:** notebook 04 covers the **XGBoost cost regressor** that ingests the classifier + detector outputs and emits dollars.
